# Giảm chiều MNIST – Notebook chung (PCA / Chi-Square)

Một biến quyết định phương pháp; mọi phân nhánh nằm trong module `lib.reduction`. So sánh baseline vs sau giảm chiều: bộ nhớ, thời gian, độ chính xác.

## 1. Cấu hình và import

In [ ]:
import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

from lib import (
    REDUCTION_METHODS,
    create_reducer,
    get_default_n_components_trials,
    MNISTDataLoader,
    MNISTClassifier,
    measure_array_memory_mb,
    run_and_measure_seconds,
    plot_comparison_reduction,
    print_comparison_table,
    plot_confusion_matrix,
    print_classification_report,
)
import matplotlib.pyplot as plt

In [ ]:
METHOD = "pca"
N_COMPONENTS = 0.95
RANDOM_STATE = 42

print(f"Phương pháp: {METHOD}, n_components: {N_COMPONENTS}. \nCó thể đổi METHOD thành một trong {REDUCTION_METHODS}.")

## 2. Tải dữ liệu

In [ ]:
loader = MNISTDataLoader(normalize=True, flatten=True)
X_train, y_train, X_test, y_test = loader.load()

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Số chiều gốc: {X_train.shape[1]}")

## 3. Baseline – Dữ liệu gốc

In [ ]:
baseline_memory_mb = measure_array_memory_mb(X_train)
print(f"Bộ nhớ X_train (baseline): {baseline_memory_mb:.4f} MB")

In [ ]:
clf_baseline = MNISTClassifier(model_type="logistic", random_state=RANDOM_STATE)

_, time_fit_baseline = run_and_measure_seconds(lambda: clf_baseline.fit(X_train, y_train))
_, time_predict_baseline = run_and_measure_seconds(lambda: clf_baseline.predict(X_test))

acc_baseline = clf_baseline.score(X_test, y_test)

baseline_metrics = {
    "memory_mb": baseline_memory_mb,
    "time_fit_s": time_fit_baseline,
    "time_predict_s": time_predict_baseline,
    "accuracy": acc_baseline,
}
print(f"Baseline - Fit: {time_fit_baseline:.4f} s, Predict: {time_predict_baseline:.4f} s, Accuracy: {acc_baseline:.4f}")

## 4. Giảm chiều

In [ ]:
reducer = create_reducer(METHOD, n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_train_reduced = reducer.fit_transform(X_train, y_train, X_val=X_test, y_val=y_test)
X_test_reduced = reducer.transform(X_test)

print(f"Số chiều sau giảm: {reducer.n_components_}")
print(f"Tổng phương sai (PCA): {reducer.total_explained_variance_ratio()}")
print(f"Độ chính xác đạt (Chi-Square): {reducer.min_accuracy_reached()}")

In [ ]:
reduced_memory_mb = measure_array_memory_mb(X_train_reduced)
print(f"Bộ nhớ X_train (sau giảm chiều): {reduced_memory_mb:.4f} MB")

## 5. Đánh giá sau giảm chiều

In [ ]:
clf_reduced = MNISTClassifier(model_type="logistic", random_state=RANDOM_STATE)

_, time_fit_reduced = run_and_measure_seconds(lambda: clf_reduced.fit(X_train_reduced, y_train))
_, time_predict_reduced = run_and_measure_seconds(lambda: clf_reduced.predict(X_test_reduced))

acc_reduced = clf_reduced.score(X_test_reduced, y_test)

reduced_metrics = {
    "memory_mb": reduced_memory_mb,
    "time_fit_s": time_fit_reduced,
    "time_predict_s": time_predict_reduced,
    "accuracy": acc_reduced,
}
print(f"Sau giảm chiều - Fit: {time_fit_reduced:.4f} s, Predict: {time_predict_reduced:.4f} s, Accuracy: {acc_reduced:.4f}")

## 6. So sánh và trực quan

In [ ]:
print_comparison_table(baseline_metrics, reduced_metrics, reduced_name=f"Sau giảm chiều ({reducer.method_label})")

In [ ]:
fig = plot_comparison_reduction(
    baseline_metrics,
    reduced_metrics,
    title=f"So sánh Baseline vs Sau giảm chiều ({reducer.method_label})",
)
plt.show()

## 7. Báo cáo phân lớp và Confusion Matrix

In [ ]:
y_pred_reduced = clf_reduced.predict(X_test_reduced)
class_names = loader.get_class_names()

print(f"Báo cáo phân lớp – Sau giảm chiều ({reducer.method_label}):")
print_classification_report(y_test, y_pred_reduced, class_names=class_names)

In [ ]:
fig = plot_confusion_matrix(y_test, y_pred_reduced, class_names=class_names, title=f"Confusion Matrix – {reducer.method_label}")
plt.show()

## 8. Thử nhiều mức n_components

In [ ]:
results = []
for nc in get_default_n_components_trials(METHOD):
    r = create_reducer(METHOD, n_components=nc, random_state=RANDOM_STATE)
    X_tr = r.fit_transform(X_train, y_train, X_val=X_test, y_val=y_test)
    X_te = r.transform(X_test)
    mem = measure_array_memory_mb(X_tr)
    clf = MNISTClassifier(model_type="logistic", random_state=RANDOM_STATE)
    clf.fit(X_tr, y_train)
    acc = clf.score(X_te, y_test)
    results.append({"n_components": nc, "dims": r.n_components_, "memory_mb": mem, "accuracy": acc})

print(f"METHOD={METHOD}")
print(f"{'n_components':>12} | {'Số chiều':>8} | {'Bộ nhớ (MB)':>11} | {'Accuracy':>8}")
for row in results:
    print(f"{row['n_components']:>12.4f} | {row['dims']:>8.0f} | {row['memory_mb']:>11.4f} | {row['accuracy']:>8.4f}")

## Sinh viên thực hiện:
### - Phạm Quỳnh Chi - 223001699
### - Nguyễn Bảo Long - 223001756
### - Nguyễn Gia Bách - 223001694
### - Nguyễn Huy Kiên - 223001747